In [1]:
import os
import ast
import pandas as pd

# ============================================================
# Paths
# ============================================================

topdir = "/Users/sm6511/Desktop/Prediction-Accomodation-Exp"

outputdir = os.path.join(
    topdir,
    "Analysis/validation/explanations"
)

os.makedirs(outputdir, exist_ok=True)


# ============================================================
# Loop through Studies 1–5
# ============================================================

for study_num in range(1, 6):

    study = f"Study{study_num}.0"

    print(f"\nProcessing {study}...")

    # --------------------------------------------------------
    # Load cleaned accommodate data
    # --------------------------------------------------------

    cleandir = os.path.join(
        topdir,
        f"data/{study}/Cleaned"
    )

    accommodate_path = os.path.join(
        cleandir,
        f"{study}Accommodate.csv"
    )

    df = pd.read_csv(accommodate_path)

    # Make sure participant / condition-order ID is numeric
    df["participant"] = pd.to_numeric(
        df["participant"],
        errors="coerce"
    )

    # Remove rows without participant IDs
    df = df.dropna(subset=["participant"]).copy()

    df["participant"] = df["participant"].astype(int)

    # Sort by condition order
    df = df.sort_values("participant")


    # --------------------------------------------------------
    # Output file
    # --------------------------------------------------------

    output_path = os.path.join(
        outputdir,
        f"{study}_explanations.txt"
    )


    # --------------------------------------------------------
    # Extract and write explanations
    # --------------------------------------------------------

    with open(output_path, "w", encoding="utf-8") as f:

        for _, row in df.iterrows():

            participant = row["participant"]
            free_texts = row["free_texts"]

            # Participant / condition-order heading
            f.write(f"{participant}.\n")

            # Handle missing values
            if pd.isna(free_texts):
                f.write("[NO FREE TEXT]\n\n")
                continue

            # free_texts is usually stored as a stringified list
            if isinstance(free_texts, str):
                try:
                    free_texts = ast.literal_eval(free_texts)
                except (ValueError, SyntaxError):
                    # If for some reason it isn't a valid list,
                    # just treat the whole cell as one explanation
                    free_texts = [free_texts]

            # Make sure it is iterable as a list
            if not isinstance(free_texts, (list, tuple)):
                free_texts = [free_texts]

            # Write each explanation on its own line
            for text in free_texts:
                f.write(f"{text}\n")

            # Blank line between participants
            f.write("\n")

    print(f"Saved: {output_path}")


print("\nDone.")


Processing Study1.0...
Saved: /Users/sm6511/Desktop/Prediction-Accomodation-Exp/Analysis/validation/explanations/Study1.0_explanations.txt

Processing Study2.0...
Saved: /Users/sm6511/Desktop/Prediction-Accomodation-Exp/Analysis/validation/explanations/Study2.0_explanations.txt

Processing Study3.0...
Saved: /Users/sm6511/Desktop/Prediction-Accomodation-Exp/Analysis/validation/explanations/Study3.0_explanations.txt

Processing Study4.0...
Saved: /Users/sm6511/Desktop/Prediction-Accomodation-Exp/Analysis/validation/explanations/Study4.0_explanations.txt

Processing Study5.0...
Saved: /Users/sm6511/Desktop/Prediction-Accomodation-Exp/Analysis/validation/explanations/Study5.0_explanations.txt

Done.


In [2]:
import os
import ast
import math
import pandas as pd

# ============================================================
# Settings
# ============================================================

topdir = "/Users/sm6511/Desktop/Prediction-Accomodation-Exp"

outputdir = os.path.join(
    topdir,
    "Analysis/validation/explanations"
)

sample_fraction = 0.10
random_seed = 42


# ============================================================
# Randomly sample 2% from each study
# ============================================================

sampled_blocks = []

for study_num in range(1, 6):

    study = f"Study{study_num}.0"

    cleandir = os.path.join(
        topdir,
        f"data/{study}/Cleaned"
    )

    accommodate_path = os.path.join(
        cleandir,
        f"{study}Accommodate.csv"
    )

    df = pd.read_csv(accommodate_path)

    # Clean participant IDs
    df["participant"] = pd.to_numeric(
        df["participant"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["participant", "free_texts"]
    ).copy()

    df["participant"] = df["participant"].astype(int)

    # --------------------------------------------------------
    # Number to sample
    #
    # ceil ensures that small studies still contribute
    # at least one case when 2% > 0
    # --------------------------------------------------------

    n_sample = max(
        1,
        math.ceil(len(df) * sample_fraction)
    )

    print(
        f"{study}: sampling {n_sample} "
        f"of {len(df)} participants"
    )

    sampled_df = df.sample(
        n=n_sample,
        random_state=random_seed + study_num
    )

    # Sort sampled cases by original condition-order ID
    sampled_df = sampled_df.sort_values("participant")

    # --------------------------------------------------------
    # Store sampled explanation blocks
    # --------------------------------------------------------

    for _, row in sampled_df.iterrows():

        participant = row["participant"]
        free_texts = row["free_texts"]

        if isinstance(free_texts, str):
            try:
                free_texts = ast.literal_eval(free_texts)
            except (ValueError, SyntaxError):
                free_texts = [free_texts]

        if not isinstance(free_texts, (list, tuple)):
            free_texts = [free_texts]

        sampled_blocks.append(
            {
                "study": study,
                "participant": participant,
                "free_texts": free_texts
            }
        )


# ============================================================
# Save all sampled explanations into one file
# ============================================================

output_path = os.path.join(
    outputdir,
    "Random_2pct_Explanation_Validation_Sample.txt"
)

with open(output_path, "w", encoding="utf-8") as f:

    for block in sampled_blocks:

        study = block["study"]
        participant = block["participant"]
        free_texts = block["free_texts"]

        # Study + original condition-order number
        f.write(f"{study} | {participant}.\n")

        for text in free_texts:
            f.write(f"{text}\n")

        f.write("\n")


print("\nSaved combined validation sample to:")
print(output_path)

Study1.0: sampling 15 of 150 participants
Study2.0: sampling 21 of 209 participants
Study3.0: sampling 22 of 217 participants
Study4.0: sampling 21 of 204 participants
Study5.0: sampling 30 of 292 participants

Saved combined validation sample to:
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/Analysis/validation/explanations/Random_2pct_Explanation_Validation_Sample.txt
